In [12]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

# -------------------------------------------------------------------
# 1. LUT FILE PATH CONFIGURATION & DATA LOADER
# -------------------------------------------------------------------
# Set the directory path where your .npz LUT files are stored
LUT_FOLDER_PATH = "../../../ihp-gmid-kit/data"  # Example: "./data" or "./ihp-gmid-kit/data"
NMOS_LUT_FILE = os.path.join(LUT_FOLDER_PATH, "sg13_lv_nmos.npz")
PMOS_LUT_FILE = os.path.join(LUT_FOLDER_PATH, "sg13_lv_pmos.npz")

class GMID_LUT_Engine:
    def __init__(self, nmos_path, pmos_path):
        self.lut_data = {}
        self.is_loaded = False
        
        # Attempt to load .npz files
        if os.path.exists(nmos_path) and os.path.exists(pmos_path):
            try:
                self.lut_data['NMOS'] = np.load(nmos_path, allow_pickle=True)
                self.lut_data['PMOS'] = np.load(pmos_path, allow_pickle=True)
                self.is_loaded = True
                print("[SUCCESS] Successfully loaded IHP SG13CMOS5L LUT files!")
            except Exception as e:
                print(f"[ERROR] Failed to parse LUT files: {e}")
        else:
            print("[WARNING] Could not locate .npz LUT files at specified paths!")
            print(f"          Checked: {nmos_path} and {pmos_path}")
            print("          Falling back to analytical physical estimation model.")
            
    def lookup(self, device_type, param_name, gm_id_val, L_val, Vds_val):
        """
        Query parameters with distinct NMOS vs. PMOS models.
        """
        if self.is_loaded and device_type in self.lut_data:
            data = self.lut_data[device_type]
            try:
                gm_id_grid = data['gm_id'] if 'gm_id' in data else data['GM_ID']
                
                if param_name == 'ID_W':
                    id_w_array = data['id_w'] if 'id_w' in data else data['ID_W']
                    val = np.interp(gm_id_val, gm_id_grid, id_w_array)
                    return float(val)
                elif param_name == 'GM_GDS':
                    gm_gds_array = data['gm_gds'] if 'gm_gds' in data else data['GM_GDS']
                    val = np.interp(gm_id_val, gm_id_grid, gm_gds_array)
                    return float(val)
            except Exception:
                pass
    
        # --- DEVICE-SPECIFIC FALLBACK (When .npz is not loaded/matched) ---
        # PMOS mobility (mu_p) is roughly 2.5x to 3x lower than NMOS (mu_n)
        mobility_factor = 1.0 if device_type == 'NMOS' else 0.38  # PMOS current density is ~38% of NMOS
        gain_factor = 1.0 if device_type == 'NMOS' else 1.15       # PMOS often has slightly higher ro
    
        if param_name == 'ID_W':
            # Apply mobility penalty for PMOS
            id_w = (220.0 / (gm_id_val**1.85)) * (0.13 / L_val)**0.35 * (Vds_val / 0.6)**0.12 * 1e-6
            return id_w * mobility_factor
        elif param_name == 'GM_GDS':
            gm_gds = (3.6 * gm_id_val) * (L_val / 0.13)**0.82
            return gm_gds * gain_factor
    
        return 1.0

# Initialize the LUT Engine
lut_engine = GMID_LUT_Engine(NMOS_LUT_FILE, PMOS_LUT_FILE)

[SUCCESS] Successfully loaded IHP SG13CMOS5L LUT files!


In [13]:
# -------------------------------------------------------------------
# 2. INTERACTIVE WIDGET CONTROLS
# -------------------------------------------------------------------
style = {'description_width': '200px'}
layout = widgets.Layout(width='460px')

w_device = widgets.Dropdown(options=['NMOS', 'PMOS'], value='NMOS', description='Device Type:', style=style, layout=layout)
w_gbw = widgets.FloatText(value=13.0, description='Target GBW (MHz):', style=style, layout=layout)
w_irn = widgets.FloatText(value=10.0, description='Target IRN (nV/√Hz):', style=style, layout=layout)
w_cl = widgets.FloatText(value=0.80, description='Load Capacitance CL (pF):', style=style, layout=layout)
w_gmid_target = widgets.FloatSlider(value=16.0, min=5.0, max=22.0, step=0.5, description='Target gm/ID (V^-1):', style=style, layout=layout)
w_L = widgets.Dropdown(options=[0.13, 0.18, 0.25, 0.35, 0.50, 0.75, 1.0, 1.5, 2.0], value=0.50, description='Channel Length L (µm):', style=style, layout=layout)
w_Vds = widgets.FloatSlider(value=0.6, min=0.1, max=1.2, step=0.05, description='Drain-Source VDS (V):', style=style, layout=layout)
w_gamma = widgets.FloatText(value=1.33, description='Thermal Noise Factor (γ):', style=style, layout=layout)

output_panel = widgets.Output()

In [14]:
# -------------------------------------------------------------------
# 3. COMPUTATION & INTERACTIVE PLOTTING
# -------------------------------------------------------------------
def update_design_estimate(change=None):
    device_type = w_device.value
    gbw_hz = w_gbw.value * 1e6
    irn_val = w_irn.value * 1e-9
    cl_farad = w_cl.value * 1e-12
    gmid_target = w_gmid_target.value
    L_val = w_L.value
    Vds_val = w_Vds.value
    gamma = w_gamma.value
    kT = 4.11e-21  # Boltzmann constant * T at 300K

    # 1. Compute transconductance (gm) constraints
    gm_gbw = 2 * np.pi * gbw_hz * cl_farad
    gm_irn = (8/3 * gamma * kT) / (irn_val**2) if irn_val > 0 else 0
    gm_design = max(gm_gbw, gm_irn)

    # 2. Compute required bias current (ID)
    ID_req = gm_design / gmid_target

    # 3. Query LUT dataset (.npz)
    id_over_w = lut_engine.lookup(device_type, 'ID_W', gmid_target, L_val, Vds_val)
    gm_gds = lut_engine.lookup(device_type, 'GM_GDS', gmid_target, L_val, Vds_val)

    # 4. Transistor sizing and output performance extraction
    W_req = ID_req / id_over_w if id_over_w > 0 else 0
    gds_req = gm_design / gm_gds if gm_gds > 0 else 0
    ro_req = 1.0 / gds_req if gds_req > 0 else 0
    vov_est = 2.0 / gmid_target  # Overdrive voltage approximation: 2 / (gm/ID)

    with output_panel:
        output_panel.clear_output(wait=True)

        # HTML Summary Table
        table_html = f"""
        <div style="font-family: Arial, sans-serif; max-width: 700px;">
            <h3 style="color: #2B579A; border-bottom: 2px solid #2B579A; margin-bottom: 8px;">
                Design Estimation Results (SG13CMOS5L {device_type}, L = {L_val} µm)
            </h3>
            
            <table style="width:100%; border-collapse: collapse; font-size: 13px;">
                <thead>
                    <tr style="background-color: #2B579A; color: white;">
                        <th style="padding: 6px; text-align: left;">Performance Parameter</th>
                        <th style="padding: 6px; text-align: right;">Estimated Value</th>
                        <th style="padding: 6px; text-align: left;">Unit</th>
                    </tr>
                </thead>
                <tbody>
                    <tr style="background-color: #f9f9f9;">
                        <td style="padding: 6px;"><b>Design Transconductance (g_m)</b></td>
                        <td style="padding: 6px; text-align: right;"><b>{gm_design * 1e6:.2f}</b></td>
                        <td style="padding: 6px;">µS</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px;"><b>Required Bias Current (I_D)</b></td>
                        <td style="padding: 6px; text-align: right;"><b>{ID_req * 1e6:.2f}</b></td>
                        <td style="padding: 6px;">µA</td>
                    </tr>
                    <tr style="background-color: #fff2cc;">
                        <td style="padding: 6px; color: #B25900;"><b>Transistor Width (W)</b></td>
                        <td style="padding: 6px; text-align: right; color: #B25900; font-size: 15px;"><b>{W_req:.2f}</b></td>
                        <td style="padding: 6px; color: #B25900;"><b>µm</b></td>
                    </tr>
                    <tr>
                        <td style="padding: 6px;"><b>Current Density (I_D / W)</b></td>
                        <td style="padding: 6px; text-align: right;">{(id_over_w * 1e6):.3f}</td>
                        <td style="padding: 6px;">µA / µm</td>
                    </tr>
                    <tr style="background-color: #f9f9f9;">
                        <td style="padding: 6px;"><b>Output Conductance (g_ds)</b></td>
                        <td style="padding: 6px; text-align: right;">{gds_req * 1e6:.3f}</td>
                        <td style="padding: 6px;">µS</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px;"><b>Output Resistance (r_o)</b></td>
                        <td style="padding: 6px; text-align: right;">{(ro_req * 1e-3):.2f}</td>
                        <td style="padding: 6px;">kΩ</td>
                    </tr>
                    <tr style="background-color: #e2efda;">
                        <td style="padding: 6px; color: #276A3C;"><b>Intrinsic Gain (g_m / g_ds)</b></td>
                        <td style="padding: 6px; text-align: right; color: #276A3C;"><b>{gm_gds:.2f} ({20*np.log10(gm_gds):.2f} dB)</b></td>
                        <td style="padding: 6px; color: #276A3C;">V/V</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px;"><b>Estimated Overdrive Voltage (V_ov)</b></td>
                        <td style="padding: 6px; text-align: right;">{vov_est*1e3:.1f}</td>
                        <td style="padding: 6px;">mV</td>
                    </tr>
                </tbody>
            </table>
        </div>
        """
        display(HTML(table_html))

        # --- INTERACTIVE GM/ID DESIGN CURVES ---
        gm_id_sweep = np.linspace(4.0, 22.0, 60)
        id_w_sweep = [lut_engine.lookup(device_type, 'ID_W', g, L_val, Vds_val) * 1e6 for g in gm_id_sweep]
        gain_sweep = [lut_engine.lookup(device_type, 'GM_GDS', g, L_val, Vds_val) for g in gm_id_sweep]

        fig, ax1 = plt.subplots(figsize=(7.5, 3.8))

        # Current density curve (ID/W)
        color1 = '#1f77b4'
        ax1.set_xlabel('gm/ID (V^-1)', fontweight='bold')
        ax1.set_ylabel('Current Density ID/W (µA/µm)', color=color1, fontweight='bold')
        ax1.plot(gm_id_sweep, id_w_sweep, color=color1, linewidth=2, label='ID/W')
        ax1.tick_params(axis='y', labelcolor=color1)
        ax1.grid(True, linestyle=':', alpha=0.6)

        # Operating point indicator
        ax1.plot(gmid_target, id_over_w * 1e6, 'ro', markersize=8, label=f'Operating Point (gm/ID={gmid_target})')

        # Intrinsic gain curve (gm/gds)
        ax2 = ax1.twinx()
        color2 = '#2ca02c'
        ax2.set_ylabel('Intrinsic Gain gm/gds (V/V)', color=color2, fontweight='bold')
        ax2.plot(gm_id_sweep, gain_sweep, color=color2, linewidth=2, linestyle='--', label='gm/gds')
        ax2.tick_params(axis='y', labelcolor=color2)

        plt.title(f'Transistor Characteristic Curves ({device_type}, L = {L_val} µm, VDS = {Vds_val} V)', fontsize=11)
        fig.tight_layout()
        plt.show()

# Attach observers to all input widgets
for widget in [w_device, w_gbw, w_irn, w_cl, w_gmid_target, w_L, w_Vds, w_gamma]:
    widget.observe(update_design_estimate, names='value')


In [15]:
# -------------------------------------------------------------------
# 4. DISPLAY DASHBOARD
# -------------------------------------------------------------------
input_panel = widgets.VBox([
    widgets.HTML("<h4>Input Parameters & Circuit Targets</h4>"),
    w_device, w_gbw, w_irn, w_cl, w_gmid_target, w_L, w_Vds, w_gamma
])

display(widgets.HBox([input_panel, output_panel]))

# Trigger initial calculation run
update_design_estimate()